In [1]:
#Setup Machine Template
!test -d spark && echo "Skipping" || (wget -q https://dlcdn.apache.org/spark/spark-4.1.1/spark-4.1.1-bin-hadoop3.tgz -O spark.tgz && mkdir spark && tar xvf spark.tgz -C spark --strip-components=1 > /dev/null && rm spark.tgz)
%pip install -q kaggle



Skipping
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_HOME"] = os.path.join(os.getcwd(), "spark")
os.environ['KAGGLE_USERNAME'] = os.getenv("K_USER", "xxxxx")
os.environ['KAGGLE_KEY'] = os.getenv("K_TOKEN", "xxxxxxx")
!test -d nytdataset && echo "Skipping" || kaggle datasets download -d "benjaminawd/new-york-times-articles-comments-2020"
!test -d nytdataset && echo "Skipping" || unzip -d nytdataset new-york-times-articles-comments-2020.zip
!rm new-york-times-articles-comments-2020.zip 2> /dev/null

Skipping
Skipping


In [3]:
import pyspark
from pyspark.sql import SparkSession
import re

REGEX = re.compile(r"[^a-z]+")
def normalizeWord(word):
    word = re.sub(REGEX, "", str(word).lower())
    return word

spark = SparkSession.builder.master("local[*]").appName("Clustering Sacchi Andrea").getOrCreate()
csv_options = {
    "header": True,
    "escape": '"'
}
df = spark.read.options(**csv_options).csv("nytdataset/nyt-articles-2020.csv")
ABSTRACT_VECTOR_LENGTH = 500
CLUSTERS = df.select("section").rdd.map(normalizeWord).distinct().count()
ITERATIONS = 20
SEED = None


# Initialization

In [ ]:


def abstract_vectorinit(row):
    words = str(row.abstract).replace("New York", "NewYork").split(" ")
    words_found = 0
    abstract_vectorized = [0.0]*ABSTRACT_VECTOR_LENGTH
    for word in words:
        word = normalizeWord(word)
        if word in bag_of_words_bcast.value:
            abstract_vectorized[bag_of_words_bcast.value[word]]  += 1.0
            words_found += 1

    if words_found > 0:
        for i in range(ABSTRACT_VECTOR_LENGTH):
            abstract_vectorized[i] /= float(words_found)
    return abstract_vectorized

STOPWORDS = spark.sparkContext.broadcast(["", "i", "me", "my", "myself", "we", "our", "ours", "ourselves", "you", "your", "yours", "yourself", "yourselves", "he", "him", "his", "himself", "she", "her", "hers", "herself", "it", "its", "itself", "they", "them", "their", "theirs", "themselves", "what", "which", "who", "whom", "this", "that", "these", "those", "am", "is", "are", "was", "were", "be", "been", "being", "have", "has", "had", "having", "do", "does", "did", "doing", "a", "an", "the", "and", "but", "if", "or", "because", "as", "until", "while", "of", "at", "by", "for", "with", "about", "against", "between", "into", "through", "during", "before", "after", "above", "below", "to", "from", "up", "down", "in", "out", "on", "off", "over", "under", "again", "further", "then", "once", "here", "there", "when", "where", "why", "how", "all", "any", "both", "each", "few", "more", "most", "other", "some", "such", "no", "nor", "not", "only", "own", "same", "so", "than", "too", "very", "s", "t", "can", "will", "just", "don", "should", "now", "said", "dont", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten", "even", "time", "could", "year"])
normalized_words_rdd = df.select("abstract").rdd \
    .flatMap(lambda x: str(x.abstract).replace("New York", "NewYork").split(" ")) \
    .map(normalizeWord) \
    .filter(lambda x: x not in STOPWORDS.value)

term_frequency = normalized_words_rdd \
    .map(lambda x: (x, 1)) \
    .reduceByKey(lambda x, y: x+y) \
    .sortBy(lambda x: x[1], False) \
    .take(ABSTRACT_VECTOR_LENGTH)
    
bow_map_index = {item[0]: index for index,item in enumerate(term_frequency)} 
reversed_bow_map = {index: item[0] for index,item in enumerate(term_frequency)} 
bag_of_words_bcast = spark.sparkContext.broadcast(bow_map_index)
abstract_vectors = df.select("abstract").rdd.map(abstract_vectorinit).filter(lambda x: sum(x) > 0).persist(pyspark.StorageLevel.MEMORY_AND_DISK_DESER)
bag_of_words_bcast.unpersist()
    

# KMeans implementation

In [ ]:
#K-Means implementation

class KMeans():
    
    @staticmethod
    def _euclideanDistance(v1, v2):
        return sum((a - b) * (a - b) for a,b in zip(v1, v2)) ** 0.5
    
    @staticmethod
    def _closestCentroid(x, centroids, distance):
        best_i = 0
        min_dist = float('inf')
        for i, c in enumerate(centroids):
            d = distance(c, x)
            if d < min_dist:
                min_dist = d
                best_i = i
        return best_i, min_dist

    def __init__(self, k, iterations, rdd, distance=_euclideanDistance, seed=None, debug=False):
        self.k_target = k
        self.iteration = 0
        self.iterations = iterations
        self.centroids = [rdd.sample(False, 0.1, seed=seed).first()]
        self.rdd : pyspark.rdd.RDD = rdd
        self.distance = distance
        self.debug = debug

    def step(self):
        if self.iteration >= self.iterations: 
            return False
        distance = self.distance
        closestCentroid = KMeans._closestCentroid
        centroids = spark.sparkContext.broadcast(self.centroids)
        if self.iteration == 0 and len(self.centroids) < self.k_target:
            new_centroid = self.rdd.map(lambda x: (x, closestCentroid(x, centroids.value, distance))).max(key=lambda x: x[1][1])
            self.centroids.append(new_centroid[0])
        else: 
            self.iteration+=1 
            new_centroids = self.rdd.map(lambda x: (x, closestCentroid(x, centroids.value, distance))) \
            .map(lambda x: (x[1][0], (x[0], 1))) \
            .reduceByKey(lambda x, y: ([a + b for a, b in zip(x[0], y[0])], x[1] + y[1])) \
            .collectAsMap()

            for c in range(self.k_target):
                for i in range(len(new_centroids[c][0])):
                    new_centroids[c][0][i] /= new_centroids[c][1]
                if self.debug:
                    print(f"Centroid {c} moved: ", distance(self.centroids[c], new_centroids[c][0]))
                self.centroids[c] = new_centroids[c][0] 
        centroids.unpersist()
        return True

    

In [6]:
kmeans = KMeans(k=CLUSTERS, iterations=ITERATIONS, rdd=abstract_vectors, seed=SEED, debug=True)
while kmeans.step():
    print(kmeans.iteration)
    pass

0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
Centroid 0 moved:  0.9744235555129841
Centroid 1 moved:  0.8036441457969453
Centroid 2 moved:  0.7910574192100415
Centroid 3 moved:  0.7492219115374015
Centroid 4 moved:  0.6966149248541362
Centroid 5 moved:  0.7438707179510529
Centroid 6 moved:  0.7184548951533964
Centroid 7 moved:  0.7336457116124163
Centroid 8 moved:  0.7721032612640322
Centroid 9 moved:  0.763930354449029
Centroid 10 moved:  0.74472390376911
Centroid 11 moved:  0.7156585124592009
Centroid 12 moved:  0.7314318452051471
Centroid 13 moved:  0.754533707356783
Centroid 14 moved:  0.7670241956293985
Centroid 15 moved:  0.7364280649039282
Centroid 16 moved:  0.7267624496048115
Centroid 17 moved:  0.7657408777165269
Centroid 18 moved:  0.7671360316736227
Centroid 19 moved:  0.7307280294679047
Centroid 20 moved:  0.6582562256530824
Centroid 21 moved:  0.7786984440827713
Centroid 22 moved:  0.7288381828497622
Centroid 23 moved:  0.7820855608601

# Experiments
I'm recreating the abstract vectorization with the addition of the original real section of each article

In [7]:
def abstract_vector_init_with_sections(row): 
    words = str(row[1]).replace("New York", "NewYork").split(" ")
    words_found = 0
    abstract_vectorized = [0.0]*ABSTRACT_VECTOR_LENGTH
    for word in words:
        word = normalizeWord(word)
        if word in bag_of_words_bcast.value:
            abstract_vectorized[bag_of_words_bcast.value[word]]  += 1.0
            words_found += 1

    if words_found > 0:
        for i in range(ABSTRACT_VECTOR_LENGTH):
            abstract_vectorized[i] /= float(words_found)
    return (normalizeWord(row[0]), abstract_vectorized)

abstract_vectors_with_sections = df.select("section", "abstract").rdd.map(abstract_vector_init_with_sections).filter(lambda x: sum(x[1]) > 0)

final_centroids_bcast = spark.sparkContext.broadcast(kmeans.centroids)
distance = KMeans._euclideanDistance
def assign_point_to_cluster(data):
    real_section = data[0]
    abstract_vector = data[1]
    
    distances = [(i, distance(c, abstract_vector)) for i, c in enumerate(final_centroids_bcast.value)]

    cluster_id = min(distances, key=lambda x: x[1])[0]
    
    return (cluster_id, real_section)

predictions_rdd = abstract_vectors_with_sections.map(assign_point_to_cluster)
df_results = predictions_rdd.toDF(["Cluster_ID", "Section"])
crosstab_df = df_results.crosstab("Cluster_ID", "Section")

Top 10 word for each cluster centroid

In [8]:
for i, c in enumerate(kmeans.centroids):
    print(f"Cluster {i}:", list(map(lambda x: reversed_bow_map[x[0]], sorted(enumerate(c), key=lambda x: x[1], reverse=True)[:10])))

Cluster 0: ['new', 'coronavirus', 'pandemic', 'people', 'newyork', 'many', 'city', 'home', 'house', 'make']
Cluster 1: ['national', 'security', 'former', 'president', 'first', 'would', 'book', 'new', 'newyork', 'coronavirus']
Cluster 2: ['help', 'pandemic', 'new', 'make', 'coronavirus', 'need', 'us', 'get', 'people', 'may']
Cluster 3: ['going', 'away', 'like', 'youre', 'keep', 'pandemic', 'love', 'heres', 'years', 'making']
Cluster 4: ['us', 'puzzle', 'coronavirus', 'bring', 'take', 'help', 'joe', 'offers', 'many', 'find']
Cluster 5: ['months', 'last', 'report', 'percent', 'ahead', 'many', 'first', 'ago', 'heres', 'without']
Cluster 6: ['lot', 'us', 'still', 'theres', 'youre', 'power', 'new', 'small', 'first', 'made']
Cluster 7: ['good', 'way', 'us', 'news', 'new', 'thing', 'people', 'coronavirus', 'might', 'thats']
Cluster 8: ['difficult', 'coronavirus', 'times', 'made', 'states', 'make', 'family', 'would', 'pandemic', 'living']
Cluster 9: ['debate', 'pandemic', 'whether', 'president'

In [9]:
crosstab_df.show(1000, False)

+------------------+-----+----+------+-----------+-----+--------+-----------+-------+---------------+---------+------------+----+------+-----------------+--------+------+-------+----------+-------+---------+--------+------------+----------+-------+-------------+------+-----+------------+----------+-------+------------------+---------+---------+------------+---------+------+---------+----+-----+----+-----+---------+
|Cluster_ID_Section|admin|arts|athome|automobiles|books|briefing|businessday|climate|crosswordsgames|education|fashionstyle|food|health|internationalhome|magazine|movies|newyork|obituaries|opinion|parenting|podcasts|readercenter|realestate|science|smarterliving|sports|style|sundayreview|technology|theater|thelearningnetwork|theupshot|theweekly|timesinsider|tmagazine|travel|universal|us  |video|well|world|yourmoney|
+------------------+-----+----+------+-----------+-----+--------+-----------+-------+---------------+---------+------------+----+------+-----------------+--------

In [10]:
top_sections_rdd = predictions_rdd \
    .map(lambda x: ((x[0], x[1]), 1)) \
    .reduceByKey(lambda a, b: a + b) \
    .map(lambda x: (x[0][0], (x[1], x[0][1]))) \
    .reduceByKey(lambda a, b: a if a[0] > b[0] else b) \
    .map(lambda x: (x[0], x[1][1]))

cluster_to_section = top_sections_rdd.collectAsMap()

for cluster_id, section in sorted(cluster_to_section.items()):
    print(f"Cluster {cluster_id} -> {section}")


Cluster 0 -> us
Cluster 1 -> us
Cluster 2 -> opinion
Cluster 3 -> opinion
Cluster 4 -> crosswordsgames
Cluster 5 -> crosswordsgames
Cluster 6 -> opinion
Cluster 7 -> opinion
Cluster 8 -> us
Cluster 9 -> us
Cluster 10 -> opinion
Cluster 11 -> thelearningnetwork
Cluster 12 -> opinion
Cluster 13 -> arts
Cluster 14 -> world
Cluster 15 -> opinion
Cluster 16 -> sports
Cluster 17 -> well
Cluster 18 -> us
Cluster 19 -> opinion
Cluster 20 -> opinion
Cluster 21 -> us
Cluster 22 -> timesinsider
Cluster 23 -> us
Cluster 24 -> opinion
Cluster 25 -> arts
Cluster 26 -> businessday
Cluster 27 -> sports
Cluster 28 -> opinion
Cluster 29 -> newyork
Cluster 30 -> thelearningnetwork
Cluster 31 -> world
Cluster 32 -> opinion
Cluster 33 -> opinion
Cluster 34 -> arts
Cluster 35 -> opinion
Cluster 36 -> opinion
Cluster 37 -> us
Cluster 38 -> opinion
Cluster 39 -> world
Cluster 40 -> world
Cluster 41 -> us


In [11]:
for cluster_id, section in sorted(cluster_to_section.items()):
    points = predictions_rdd.filter(lambda x: x[0] == cluster_id).take(10)
    print(f"Cluster {cluster_id} assigned to: {section}")
    print(f"Original categories for 10 points in Cluster {cluster_id}:")
    for point in points:
        print(f" - {point[1]}")
    print()

Cluster 0 assigned to: us
Original categories for 10 points in Cluster 0:
 - crosswordsgames
 - science
 - world
 - well
 - realestate
 - businessday
 - realestate
 - realestate
 - movies
 - newyork

Cluster 1 assigned to: us
Original categories for 10 points in Cluster 1:
 - arts
 - us
 - us
 - newyork
 - us
 - us
 - world
 - sports
 - sports
 - sports

Cluster 1 assigned to: us
Original categories for 10 points in Cluster 1:
 - arts
 - us
 - us
 - newyork
 - us
 - us
 - world
 - sports
 - sports
 - sports

Cluster 2 assigned to: opinion
Original categories for 10 points in Cluster 2:
 - style
 - theupshot
 - books
 - magazine
 - businessday
 - opinion
 - opinion
 - movies
 - magazine
 - world

Cluster 2 assigned to: opinion
Original categories for 10 points in Cluster 2:
 - style
 - theupshot
 - books
 - magazine
 - businessday
 - opinion
 - opinion
 - movies
 - magazine
 - world

Cluster 3 assigned to: opinion
Original categories for 10 points in Cluster 3:
 - opinion
 - opinion
 - 

In [12]:
from pyspark.ml.linalg import Vectors
from pyspark.sql import Row
from pyspark.ml.clustering import KMeans as SparkKMeans

vector_rdd = abstract_vectors.map(lambda x: Row(features=Vectors.dense(x)))
df_features = spark.createDataFrame(vector_rdd)
official_kmeans = SparkKMeans(k=CLUSTERS, maxIter=ITERATIONS, seed=SEED, featuresCol="features")
model = official_kmeans.fit(df_features)

official_centroids = model.clusterCenters()



In [13]:
print("Original spark KMeans")
for cluster_id, centroid in enumerate(official_centroids):
    top_10_indices = sorted(range(len(centroid)), key=lambda i: centroid[i], reverse=True)[:10]
    top_10_words = [reversed_bow_map[idx] for idx in top_10_indices]
    
    print(f"Cluster {cluster_id}: {top_10_words}")



Original spark KMeans
Cluster 0: ['new', 'world', 'years', 'first', 'states', 'students', 'home', 'state', 'work', 'virus']
Cluster 1: ['pandemic', 'coronavirus', 'new', 'home', 'world', 'us', 'economic', 'students', 'health', 'amid']
Cluster 2: ['week', 'many', 'well', 'last', 'way', 'puzzle', 'island', 'died', 'comes', 'shows']
Cluster 3: ['weeks', 'properties', 'response', 'homes', 'side', 'west', 'house', 'city', 'place', 'south']
Cluster 4: ['black', 'lives', 'people', 'white', 'police', 'new', 'man', 'history', 'women', 'americans']
Cluster 5: ['mr', 'trump', 'biden', 'newyork', 'sanders', 'president', 'years', 'former', 'life', 'behind']
Cluster 6: ['back', 'go', 'look', 'find', 'new', 'online', 'events', 'get', 'today', 'life']
Cluster 7: ['political', 'america', 'protests', 'latest', 'theres', 'force', 'new', 'trump', 'years', 'presidents']
Cluster 8: ['many', 'pandemic', 'people', 'new', 'coronavirus', 'us', 'say', 'women', 'may', 'health']
Cluster 9: ['us', 'puzzle', 'bring'

In [14]:
labeled_rows = abstract_vectors_with_sections.map(
    lambda x: Row(True_Section=x[0], features=Vectors.dense(x[1]))
)
df_labeled = spark.createDataFrame(labeled_rows)
predictions_df = model.transform(df_labeled)
crosstab_official = predictions_df.crosstab("prediction", "True_Section")
crosstab_official.show(1000)

+-----------------------+-----+----+------+-----------+-----+--------+-----------+-------+---------------+---------+------------+----+------+-----------------+--------+------+-------+----------+-------+---------+--------+------------+----------+-------+-------------+------+-----+------------+----------+-------+------------------+---------+---------+------------+---------+------+---------+----+-----+----+-----+---------+
|prediction_True_Section|admin|arts|athome|automobiles|books|briefing|businessday|climate|crosswordsgames|education|fashionstyle|food|health|internationalhome|magazine|movies|newyork|obituaries|opinion|parenting|podcasts|readercenter|realestate|science|smarterliving|sports|style|sundayreview|technology|theater|thelearningnetwork|theupshot|theweekly|timesinsider|tmagazine|travel|universal|  us|video|well|world|yourmoney|
+-----------------------+-----+----+------+-----------+-----+--------+-----------+-------+---------------+---------+------------+----+------+-----------

In [15]:
top_sections_official_rdd = predictions_df.rdd \
    .map(lambda row: ((row.prediction, row.True_Section), 1)) \
    .reduceByKey(lambda a, b: a + b) \
    .map(lambda x: (x[0][0], (x[1], x[0][1]))) \
    .reduceByKey(lambda a, b: a if a[0] > b[0] else b) \
    .map(lambda x: (x[0], x[1][1]))

cluster_to_section_official = top_sections_official_rdd.collectAsMap()

print("Original spark KMeans Cluster Assignments:")
for cluster_id, section in sorted(cluster_to_section_official.items()):
    print(f"Cluster {cluster_id} -> {section}")

Original spark KMeans Cluster Assignments:
Cluster 0 -> us
Cluster 1 -> us
Cluster 2 -> crosswordsgames
Cluster 3 -> realestate
Cluster 4 -> opinion
Cluster 5 -> us
Cluster 6 -> opinion
Cluster 7 -> opinion
Cluster 8 -> us
Cluster 9 -> crosswordsgames
Cluster 10 -> opinion
Cluster 11 -> us
Cluster 12 -> us
Cluster 13 -> us
Cluster 14 -> opinion
Cluster 15 -> well
Cluster 16 -> well
Cluster 17 -> well
Cluster 18 -> opinion
Cluster 19 -> opinion
Cluster 20 -> books
Cluster 21 -> opinion
Cluster 22 -> opinion
Cluster 23 -> businessday
Cluster 24 -> opinion
Cluster 25 -> realestate
Cluster 26 -> opinion
Cluster 27 -> theater
Cluster 28 -> opinion
Cluster 29 -> food
Cluster 30 -> newyork
Cluster 31 -> opinion
Cluster 32 -> us
Cluster 33 -> world
Cluster 34 -> crosswordsgames
Cluster 35 -> food
Cluster 36 -> opinion
Cluster 37 -> businessday
Cluster 38 -> world
Cluster 39 -> opinion
Cluster 40 -> opinion
Cluster 41 -> arts


In [16]:
for cluster_id, section in sorted(cluster_to_section_official.items()):
    points = predictions_df.filter(predictions_df.prediction == cluster_id).take(10)
    print(f"Cluster {cluster_id} assigned to: {section}")
    print(f"Original categories for 10 points in Cluster {cluster_id}:")
    for point in points:
        print(f" - {point.True_Section}")
    print()

Cluster 0 assigned to: us
Original categories for 10 points in Cluster 0:
 - science
 - science
 - magazine
 - magazine
 - magazine
 - arts
 - realestate
 - realestate
 - theater
 - movies

Cluster 1 assigned to: us
Original categories for 10 points in Cluster 1:
 - world
 - arts
 - opinion
 - podcasts
 - thelearningnetwork
 - businessday
 - style
 - businessday
 - podcasts
 - world

Cluster 1 assigned to: us
Original categories for 10 points in Cluster 1:
 - world
 - arts
 - opinion
 - podcasts
 - thelearningnetwork
 - businessday
 - style
 - businessday
 - podcasts
 - world

Cluster 2 assigned to: crosswordsgames
Original categories for 10 points in Cluster 2:
 - us
 - crosswordsgames
 - arts
 - style
 - travel
 - crosswordsgames
 - world
 - style
 - us
 - style

Cluster 2 assigned to: crosswordsgames
Original categories for 10 points in Cluster 2:
 - us
 - crosswordsgames
 - arts
 - style
 - travel
 - crosswordsgames
 - world
 - style
 - us
 - style

Cluster 3 assigned to: realestat

In [ ]:
abstract_vectors.unpersist()